# 01 - Explore ERA5 Subset

This notebook demonstrates how to use the pipeline modules interactively.

Steps:
1. Load configuration
2. Download a GRIB file from Azure
3. Open it lazily with xarray
4. Inspect available variables and coordinates
5. Apply subset filters
6. Preview as pandas DataFrame

In [1]:
import sys
from pathlib import Path

# Add project root to path so we can import from src/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

In [5]:
from src.config import load_config

config = load_config()
print("Years:", config.selection.blob_files)
print("Variables:", config.selection.variables)
print("Time range:", config.selection.time_start, "to", config.selection.time_end)
print("Bounding box: lat", config.selection.lat_min, "to", config.selection.lat_max,
      "lon", config.selection.lon_min, "to", config.selection.lon_max)

2026-05-07 10:31:25,072 [INFO] src.config: Loaded .env from C:\Users\qsoerohardjo\Documents\GitHub\datalab-v\.env
2026-05-07 10:31:25,080 [INFO] src.config: Loaded config from C:\Users\qsoerohardjo\Documents\GitHub\datalab-v\config.yaml


Years: ['2018-2019']
Variables: ['t2m', 'd2m']
Time range: 2018-01-01 to 2018-03-31
Bounding box: lat -5.0 to 15.0 lon 25.0 to 45.0


## Download GRIB from Azure

This downloads the configured years to `data/raw/`. Skips if already cached.

In [6]:
from src.storage import download_selected_blobs

local_paths = download_selected_blobs(config)
print("Downloaded files:", local_paths)

2026-05-07 10:31:37,264 [INFO] src.storage: Authenticated via connection string
2026-05-07 10:31:37,267 [INFO] azure.core.pipeline.policies.http_logging_policy: Request URL: 'https://era5project.blob.core.windows.net/grib/2018-2019.grib'
Request method: 'HEAD'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.13.7 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '29cf7205-49ef-11f1-97ba-c4d0e3ae2f9e'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-05-07 10:31:38,503 [INFO] azure.core.pipeline.policies.http_logging_policy: Response status: 200
Response headers:
    'Content-Length': '16854310080'
    'Content-Type': 'application/octet-stream'
    'Last-Modified': 'Tue, 03 Mar 2026 18:42:04 GMT'
    'Accept-Ranges': 'REDACTED'
    'ETag': '"0x8DE795490D6FC98"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-reque

Downloaded files: [WindowsPath('C:/Users/qsoerohardjo/Documents/GitHub/datalab-v/data/raw/2018-2019.grib')]


## Open dataset (lazy)

Nothing is loaded into memory here. xarray + dask keeps everything lazy.

In [7]:
from src.loader import open_era5_dataset

ds = open_era5_dataset(local_paths[0], config)
ds

2026-05-07 10:34:02,957 [INFO] src.loader: Opened 2018-2019.grib: 6 variables, 7 coords


<xarray.Dataset> Size: 37GB
Dimensions:     (time: 17520, latitude: 289, longitude: 301)
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 140kB 2018-01-01 ... 2019-12-31T23:00:00
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 2kB 37.0 36.75 36.5 ... -34.5 -34.75 -35.0
  * longitude   (longitude) float64 2kB -20.0 -19.75 -19.5 ... 54.5 54.75 55.0
    valid_time  (time) datetime64[ns] 140kB dask.array<chunksize=(100,), meta=np.ndarray>
Data variables:
    u10         (time, latitude, longitude) float32 6GB dask.array<chunksize=(100, 50, 50), meta=np.ndarray>
    v10         (time, latitude, longitude) float32 6GB dask.array<chunksize=(100, 50, 50), meta=np.ndarray>
    d2m         (time, latitude, longitude) float32 6GB dask.array<chunksize=(100, 50, 50), meta=np.ndarray>
    t2m         (time, latitude, longitude) float32 6GB dask.array<chunksize=(100, 50, 50), meta=np.ndarray>
    msl         (time, latitude, longitude) float32 6GB dask.array<chunksize=(100, 50, 50), meta=np.ndarray>
    sst         (time, latitude, longitude) float32 6GB dask.array<chunksize=(100, 50, 50), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-07T10:34 GRIB to CDM+CF via cfgrib-0.9.1...

## Inspect available variables and coordinates

Use this to verify which variable names are in your GRIB file.
Update `config.yaml` if the names differ from the defaults.

In [8]:
print("Variables:", list(ds.data_vars))
print("Coordinates:", list(ds.coords))
print("Dimensions:", dict(ds.dims))

Variables: ['u10', 'v10', 'd2m', 't2m', 'msl', 'sst']
Coordinates: ['number', 'time', 'step', 'surface', 'latitude', 'longitude', 'valid_time']
Dimensions: {'time': 17520, 'latitude': 289, 'longitude': 301}


C:\Users\qsoerohardjo\AppData\Local\Temp\ipykernel_13676\4097263875.py:3: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Dimensions:", dict(ds.dims))


## Apply subset filters

Filters by variables, time range, and bounding box. Still lazy.

In [9]:
from src.subset import apply_all_filters

ds_sub = apply_all_filters(ds, config)
ds_sub

2026-05-07 11:01:08,855 [INFO] src.subset: Selected variables: ['t2m', 'd2m']
2026-05-07 11:01:08,884 [INFO] src.subset: Selected time range: 2018-01-01 to 2018-03-31
2026-05-07 11:01:08,893 [INFO] src.subset: Selected bbox: lat [-5.00, 15.00], lon [25.00, 45.00]


<xarray.Dataset> Size: 113MB
Dimensions:     (time: 2160, latitude: 81, longitude: 81)
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 17kB 2018-01-01 ... 2018-03-31T23:00:00
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 648B 15.0 14.75 14.5 ... -4.5 -4.75 -5.0
  * longitude   (longitude) float64 648B 25.0 25.25 25.5 ... 44.5 44.75 45.0
    valid_time  (time) datetime64[ns] 17kB dask.array<chunksize=(100,), meta=np.ndarray>
Data variables:
    t2m         (time, latitude, longitude) float32 57MB dask.array<chunksize=(100, 12, 20), meta=np.ndarray>
    d2m         (time, latitude, longitude) float32 57MB dask.array<chunksize=(100, 12, 20), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-07T10:34 GRIB to CDM+CF via cfgrib-0.9.1...

## Optional: Resample to daily

In [10]:
from src.aggregate import resample_dataset

ds_daily = resample_dataset(ds_sub, config)
ds_daily

2026-05-07 11:01:58,723 [INFO] src.aggregate: Resampled to '1D': 2 variables


<xarray.Dataset> Size: 5MB
Dimensions:    (latitude: 81, longitude: 81, time: 90)
Coordinates:
    number     int64 8B 0
    step       timedelta64[ns] 8B 00:00:00
    surface    float64 8B 0.0
  * latitude   (latitude) float64 648B 15.0 14.75 14.5 14.25 ... -4.5 -4.75 -5.0
  * longitude  (longitude) float64 648B 25.0 25.25 25.5 ... 44.5 44.75 45.0
  * time       (time) datetime64[ns] 720B 2018-01-01 2018-01-02 ... 2018-03-31
Data variables:
    t2m        (time, latitude, longitude) float32 2MB dask.array<chunksize=(1, 12, 20), meta=np.ndarray>
    d2m        (time, latitude, longitude) float32 2MB dask.array<chunksize=(1, 12, 20), meta=np.ndarray>
Attributes: (12/31)
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      86989
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           2t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre temperature
    units:                                    K
    standard_name:                            unknown

## Convert to pandas DataFrame

This is where data is actually loaded into memory.
Only do this after subsetting to a small slice.

In [11]:
from src.export import to_dataframe

df = to_dataframe(ds_daily)
print("Shape:", df.shape)
df.head(10)

2026-05-07 11:03:31,724 [INFO] src.export: Created DataFrame: 590490 rows, 8 columns


Shape: (590490, 8)


,latitude,longitude,time,number,step,surface,t2m,d2m
0,15.0,25.0,2018-01-01,0,0 days,0.0,285.378052,268.749603
1,15.0,25.0,2018-01-02,0,0 days,0.0,284.824738,268.954742
2,15.0,25.0,2018-01-03,0,0 days,0.0,284.648102,266.977234
3,15.0,25.0,2018-01-04,0,0 days,0.0,284.015472,264.938507
4,15.0,25.0,2018-01-05,0,0 days,0.0,285.585541,267.546692
5,15.0,25.0,2018-01-06,0,0 days,0.0,286.984039,269.132172
6,15.0,25.0,2018-01-07,0,0 days,0.0,288.225098,272.525574
7,15.0,25.0,2018-01-08,0,0 days,0.0,288.309784,270.646576
8,15.0,25.0,2018-01-09,0,0 days,0.0,288.245392,273.908112
9,15.0,25.0,2018-01-10,0,0 days,0.0,288.568573,270.517792


## Export to Zarr / Parquet

Uncomment and run to save the subset to disk.

In [ ]:
# from src.export import export_dataset
# results = export_dataset(ds_daily, config)
# print("Exported:", results)